In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

In [2]:
dataset = fetch_openml('car')
df = dataset.data
df['target'] = dataset.target

/usr/local/lib/python3.13/dist-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name car exist. Versions may be fundamentally different, returning version 2. Available versions:
- version 2, status: active
  url: https://www.openml.org/search?type=data&id=991
- version 3, status: active
  url: https://www.openml.org/search?type=data&id=40975

  warn(warning_msg)


In [3]:
df

,buying,maint,doors,persons,lug_boot,safety,target
0,vhigh,vhigh,2,2,small,low,P
1,vhigh,vhigh,2,2,small,med,P
2,vhigh,vhigh,2,2,small,high,P
3,vhigh,vhigh,2,2,med,low,P
4,vhigh,vhigh,2,2,med,med,P
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,N
1724,low,low,5more,more,med,high,N
1725,low,low,5more,more,big,low,P
1726,low,low,5more,more,big,med,N


In [8]:
def chi_square_scratch(df, attribute, target):
  attr_values = df[attribute].unique()
  target_values=df[target].unique()
  chi2=0
  for attr_value in attr_values:
    for target_value in target_values:
      observed_freq = len(df[(df[attribute]==attr_value)&(df[target]==target_value)])
      attr_prob = len(df[df[attribute]==attr_value])/len(df)
      target_prob = len(df[(df[target]==target_value)])/len(df)
      expected_freq = attr_prob*target_prob*len(df)
      chi2+=(observed_freq-expected_freq)**2/expected_freq
  return chi2

In [9]:
chi_square_scratch(df,'buying','target')

76.5218545582182

In [10]:
def selectkbest_chi2_scratch(df,k):
  attributes = df.columns[:-1]
  target = df.columns[-1]
  chi2_score={}
  for attribute in attributes:
    chi2_score[attribute] = chi_square_scratch(df,attribute,target)
  sorted_chi2_score = sorted(chi2_score.items(),key=lambda x:x[1],reverse=True)
  selected_features=[f for f,_ in sorted_chi2_score[:k]]
  return selected_features,sorted_chi2_score

In [11]:
selectkbest_chi2_scratch(df, 3)

(['safety', 'persons', 'buying'],
 [('safety', 396.34438878075235),
  ('persons', 370.2912281821373),
  ('buying', 76.5218545582182),
  ('maint', 64.17072657072656),
  ('lug_boot', 29.400095727368452),
  ('doors', 8.590650627014263)])

In [15]:
from scipy.stats import chi2_contingency
table = pd.crosstab(df['buying'],df['target'])
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(76.5218545582182), pvalue=np.float64(1.7095876176351842e-16), dof=3, expected_freq=array([[129.5, 302.5],
       [129.5, 302.5],
       [129.5, 302.5],
       [129.5, 302.5]]))

In [19]:
def SelectKbest_chi2_inbuilt(df,k):
  attributes = df.columns[:-1]
  target = df.columns[-1]
  chi2_score={}
  for attribute in attributes:
    table = pd.crosstab(df[attribute],df[target])
    chi2,_,_,_=chi2_contingency(table)
    chi2_score[attribute]=chi2
  sorted_chi2_score = sorted(chi2_score.items(),key=lambda x:x[1],reverse=True)
  selected_features=[f for f,_ in sorted_chi2_score[:k]]
  return selected_features,sorted_chi2_score


In [20]:
SelectKbest_chi2_inbuilt(df,3)

(['safety', 'persons', 'buying'],
 [('safety', np.float64(396.34438878075247)),
  ('persons', np.float64(370.2912281821373)),
  ('buying', np.float64(76.5218545582182)),
  ('maint', np.float64(64.17072657072657)),
  ('lug_boot', np.float64(29.400095727368456)),
  ('doors', np.float64(8.590650627014263))])

In [21]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for column in df.columns:
  df[column] = le.fit_transform(df[column])
from sklearn.feature_selection import SelectKBest,chi2
test = SelectKBest(chi2, k=3)
test.fit(df.iloc[:,:-1],df.iloc[:,-1])
test.scores_

array([  6.39795782,   8.75055362,   5.55984556, 177.86727081,
        18.53771977,  17.64446855])

In [22]:
df.columns[:-1][test.get_support()]

Index(['persons', 'lug_boot', 'safety'], dtype='object')

In [23]:
def find_entropy(df):
  target = df.columns[-1]
  target_values = df[target].unique()
  entropy = 0
  for target_value in target_values:
    prob = len(df[df[target]==target_value])/len(df)
    entropy+=-prob*np.log(prob)
  return entropy

In [24]:
find_entropy(df)

np.float64(0.610668040691973)

In [26]:
def find_avg_info_entropy(df,attribute):
  attr_values = df[attribute].unique()
  target = df.columns[-1]
  target_values= df[target].unique()
  avg_info_entropy = 0
  for attr_value in attr_values:
    entropy_subsample = 0
    for target_value in target_values:
      num=len(df[(df[attribute]==attr_value)&(df[target]==target_value)])
      den=len(df[df[attribute]==attr_value])
      prob = num/den
      entropy_subsample=-prob*np.log(prob+1e-7)
    weight=len(df[df[attribute]==attr_value])/len(df)
    avg_info_entropy+=weight*entropy_subsample
  return avg_info_entropy

In [27]:
find_avg_info_entropy(df,'buying')

np.float64(0.34479191700544765)

In [28]:
def selectkbest_IG_scratch(df,k):
  attributes = df.columns[:-1]
  IG_score={}
  for attribute in attributes:
    IG_score[attribute] = find_entropy(df)-find_avg_info_entropy(df,attribute)
  sorted_IG_score = sorted(IG_score.items(),key=lambda x:x[1],reverse=True)
  selected_features=[f for f,_ in sorted_IG_score[:k]]
  return selected_features,sorted_IG_score

In [29]:
selectkbest_IG_scratch(df,3)

(['safety', 'persons', 'buying'],
 [('safety', np.float64(0.3746585986972647)),
  ('persons', np.float64(0.3711251089864471)),
  ('buying', np.float64(0.26587612368652536)),
  ('maint', np.float64(0.2635457332104685)),
  ('lug_boot', np.float64(0.2557282402367137)),
  ('doors', np.float64(0.2513335465816761))])

In [32]:
from sklearn.feature_selection import SelectKBest,mutual_info_classif
test1 = SelectKBest(mutual_info_classif,k=3)
test1.fit(df.iloc[:,:-1],df.iloc[:,-1])
test1.scores_


array([0.01857524, 0.02279201, 0.00501677, 0.13614265, 0.013167  ,
       0.14948162])

In [34]:
df.columns[:-1][test1.get_support()]

Index(['maint', 'persons', 'safety'], dtype='object')

In [35]:
from sklearn.metrics import mutual_info_score
mutual_info_score(df['buying'],df['target'])

np.float64(0.022942847963213067)

In [36]:
find_entropy(df)-find_avg_info_entropy(df,'buying')

np.float64(0.26587612368652536)